# Existence and Uniqueness (Picard-Lindelöf)

Interactive exploration of concept 30 from the math-foundations map.

We will:
1. Run Picard iteration on $y' = y,\ y(0) = 1$ and watch the iterates
   converge to the Taylor polynomials of $e^t$.
2. Plot the convergence (using only stdlib — ASCII plotting).
3. Exhibit two distinct solutions of $y' = y^{2/3},\ y(0) = 0$ to see
   what goes wrong when Lipschitz continuity fails.


## 1. Picard iteration for $y' = y$, $y(0) = 1$

The Picard operator is
$$T[y](t) \;=\; 1 + \int_0^t y(s)\, ds.$$
Starting from $y_0(t) \equiv 1$ we expect
$y_k(t) = \sum_{j=0}^k t^j / j!$, the $k$-th order Taylor polynomial of $e^t$.


In [ ]:
from math import factorial, exp

def picard_iterate(n_steps):
    poly = [1.0]
    history = [list(poly)]
    for _ in range(n_steps):
        integrated = [0.0] + [poly[j] / (j + 1) for j in range(len(poly))]
        integrated[0] += 1.0
        poly = integrated
        history.append(list(poly))
    return history

polys = picard_iterate(6)
for k, p in enumerate(polys):
    print(f"y_{k}(t) =", " + ".join(f"{c:.4f} t^{j}" for j, c in enumerate(p) if c))


## 2. Check against the Taylor polynomial of $e^t$

The $k$-th iterate's coefficient of $t^j$ should equal $1/j!$ for $j \le k$
and $0$ otherwise.


In [ ]:
def taylor_coefs(k):
    return [1.0 / factorial(j) for j in range(k + 1)]

for k, p in enumerate(polys):
    ref = taylor_coefs(k)
    matches = all(abs(p[j] - ref[j]) < 1e-12 for j in range(k + 1))
    print(f"k = {k}: matches Taylor coefficients? {matches}")


## 3. Convergence to $e^t$ on $[0, 2]$ — ASCII view

We evaluate $y_k(1)$ and compare to $e^1 \approx 2.71828$.


In [ ]:
for k, p in enumerate(polys):
    val = sum(c * (1.0 ** j) for j, c in enumerate(p))
    err = abs(val - exp(1.0))
    bar = "#" * max(1, int(-1 * (err and __import__('math').log10(err) or 0)))
    print(f"  y_{k}(1) = {val:.10f}   |err| = {err:.2e}   {bar}")


## 4. The non-Lipschitz example: $y' = y^{2/3}$, $y(0) = 0$

Here $f(t,y) = y^{2/3}$ has $\partial_y f = (2/3) y^{-1/3}$, which blows up
at $y = 0$. So $f$ is **not** Lipschitz at the initial condition, and
uniqueness fails. Two solutions are
$$y_a(t) \equiv 0, \qquad y_b(t) = (t/3)^3.$$


In [ ]:
def y_a(t): return 0.0
def y_b(t): return (t / 3.0) ** 3

print("  t       y_a(t)        y_b(t) = (t/3)^3")
for t in [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]:
    print(f"  {t:>4}    {y_a(t):.6f}      {y_b(t):.6f}")

# Verify y_b satisfies y' = y^{2/3} symbolically:
# d/dt (t/3)^3 = 3 (t/3)^2 * (1/3) = (t/3)^2 = ((t/3)^3)^{2/3}.
eps = 1e-9
for t in [0.5, 1.0]:
    yp = (y_b(t + eps) - y_b(t - eps)) / (2 * eps)
    rhs = y_b(t) ** (2 / 3)
    print(f"  check t={t}: y'(t)={yp:.6f}   y^(2/3)={rhs:.6f}")


## 5. A one-parameter family of solutions

For any $\tau \ge 0$, the function
$$y_\tau(t) = \begin{cases} 0 & t \le \tau \\ ((t-\tau)/3)^3 & t > \tau \end{cases}$$
is also a solution. So in fact infinitely many solutions exist. This is
why Lipschitz continuity matters for well-posed dynamics.


In [ ]:
def y_tau(t, tau):
    return 0.0 if t <= tau else ((t - tau) / 3.0) ** 3

print("Family y_tau(2.0) for various tau:")
for tau in [0.0, 0.5, 1.0, 1.5, 1.9]:
    print(f"  tau = {tau}:  y_tau(2.0) = {y_tau(2.0, tau):.6f}")


## 6. Takeaway

Picard-Lindelöf delivers a unique local solution to $y' = f(t,y)$,
$y(t_0) = y_0$ whenever $f$ is continuous and Lipschitz in $y$ on a
rectangle around $(t_0, y_0)$. The proof is just the Banach contraction
mapping theorem applied to the integral operator $T$.

When Lipschitz fails, uniqueness can break completely, as the $y^{2/3}$
example shows. This is why every flow-based generative model (flow
matching, ELF, neural ODEs) silently relies on the velocity field being
Lipschitz in the state variable.
